# Infant Cry Language Detection – Introduction & Goals

## 📌 Introduction
Babies cry to communicate their needs, but they cannot explain the reason in words.  
For parents, especially new or busy ones, it can be challenging to understand:
- Is the baby *hungry*?
- Is the baby *in pain*?
- Does the baby *need a diaper change*?
- Is the baby *uncomfortable or sleepy*?

In this project, we will use *Machine Learning* and *Deep Learning* to classify infant cries into meaningful categories based on audio recordings.

---

## 🎯 Project Goals
1. *Detect and classify baby cries* into specific needs (e.g., hunger, pain, discomfort, etc.).
2. Build a *TensorFlow-based model* that can be deployed on web/mobile apps.
3. Create a *clean, step-by-step workflow* from raw audio data to model deployment.
4. Ensure the model is robust to:
   - Background noise
   - Different recording devices
   - Variations between babies
5. Prepare the groundwork for *future health-related features*, such as:
   - Asking follow-up questions after detecting certain cry types
   - Suggesting possible health conditions

---

## ✅ Success Criteria
- Model achieves *at least 80% accuracy* on a held-out test set.
- Model inference time is *fast enough* for real-time use.
- Code is *reproducible and ready for deployment*.

---

## 📂 Dataset
We will use a dataset containing labeled baby cry audio files, each belonging to one of several categories (e.g., hunger, pain, discomfort).  
We will preprocess these audio clips into features (MFCC, Mel-spectrograms) before feeding them into the model.


# Core libraries


In [1]:


import numpy as np
# NumPy: Used for fast numerical computations, arrays, and mathematical operations.

import pandas as pd
import glob
# Pandas: For loading, cleaning, and manipulating datasets (e.g., CSV files).

# --------------------------
# Visualization libraries
# --------------------------

import matplotlib.pyplot as plt
# Matplotlib: Base Python plotting library to create graphs and visualizations.

import seaborn as sns
# Seaborn: Built on Matplotlib; makes creating beautiful, statistical visualizations easier.

# --------------------------
# Machine learning tools
# --------------------------

from sklearn.model_selection import train_test_split
# Splits data into training and testing sets for model evaluation.

from sklearn.preprocessing import LabelEncoder
# Converts text labels (e.g., 'cry', 'noise') into numeric form for ML models.

from sklearn.metrics import classification_report, confusion_matrix
# classification_report: Gives precision, recall, and F1-score.
# confusion_matrix: Shows correct and incorrect predictions.

# --------------------------
# Audio processing libraries
# --------------------------

import librosa
# Librosa: For loading audio files, extracting features (like MFCCs) for ML.

import librosa.display
# Part of Librosa: Displays waveforms, spectrograms, etc.

import soundfile as sf
# SoundFile: Reads and writes sound files (WAV, FLAC, etc.).

# --------------------------
# Deep learning (TensorFlow & Keras)
# --------------------------

import tensorflow as tf
# TensorFlow: Google’s deep learning library, used for training neural networks.

from tensorflow.keras.models import Sequential
# Sequential: Simplest way to build a neural network layer-by-layer.

from tensorflow.keras.layers import Dense, Dropout, LSTM, Conv1D, MaxPooling1D, Flatten
# Dense: Fully connected layer in neural networks.
# Dropout: Prevents overfitting by randomly turning off neurons during training.
# LSTM: Long Short-Term Memory layer for sequence/audio data.
# Conv1D: 1D Convolution layer for extracting patterns from sequences.
# MaxPooling1D: Reduces sequence length, keeps important features.
# Flatten: Converts multi-dimensional data into 1D for Dense layers.

# --------------------------
# Warnings control
# --------------------------

import warnings
warnings.filterwarnings('ignore')
# Suppresses unnecessary warning messages in the output.

# 2.Project Configuration

In [3]:

import os

# Path to dataset
DATASET_PATH = r"C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset"

# Audio settings
SAMPLE_RATE = 22050  # Common for speech/cry processing
CLIP_DURATION = 5.0  # In seconds
SAMPLES_PER_CLIP = int(SAMPLE_RATE * CLIP_DURATION)

# Label map: maps folder names to numeric labels
LABEL_MAP = {
    "belly_pain": 0,
    "burping": 1,
    "cold_hot": 2,
    "discomfort": 3,
    "hungry": 4,
    "laugh": 5,
    "lonely": 6,
    "noise": 7,
    "scared": 8,
    "silence": 9,
    "tired": 10
}

# Reverse label map (for predictions -> human readable)
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

# Verify dataset structure
print("Dataset Path:", DATASET_PATH)
print("Subfolders:", os.listdir(DATASET_PATH))
print("Number of Classes:", len(LABEL_MAP))

Dataset Path: C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset
Subfolders: ['belly_pain', 'burping', 'cold_hot', 'discomfort', 'hungry', 'laugh', 'lonely', 'noise', 'scared', 'silence', 'tired']
Number of Classes: 11


In [4]:
from pydub import AudioSegment


# Paths
DATASET_PATH = r"C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset"
OUTPUT_PATH = r"C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset_WAV"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Loop through each subfolder (e.g., "burping", "lonely")
for folder in os.listdir(DATASET_PATH):
    folder_path = os.path.join(DATASET_PATH, folder)
    
    # Skip if not a directory
    if not os.path.isdir(folder_path):
        continue
    
    output_folder = os.path.join(OUTPUT_PATH, folder)
    os.makedirs(output_folder, exist_ok=True)

    # Process ALL files in the folder (not just specific extensions)
    for file_path in glob.glob(os.path.join(folder_path, "*")):
        try:
            # Skip if already a WAV file (optional: set force_convert=True to reconvert WAVs too)
            if file_path.lower().endswith(".wav"):
                continue

            # Load audio (pydub auto-detects format)
            audio = AudioSegment.from_file(file_path)
            
            # Generate output filename (change extension to .wav)
            filename = os.path.splitext(os.path.basename(file_path))[0] + ".wav"
            output_path = os.path.join(output_folder, filename)
            
            # Export as WAV
            audio.export(output_path, format="wav")
            print(f"✅ Converted: {file_path} → {output_path}")

        except Exception as e:
            print(f"❌ Failed to convert {file_path}: {e}")

print("\n🎉 All non-WAV files converted to WAV!")

✅ Converted: C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset\belly_pain\bp-17.3gp → C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset_WAV\belly_pain\bp-17.wav
✅ Converted: C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset\belly_pain\bp-18.3gp → C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset_WAV\belly_pain\bp-18.wav
✅ Converted: C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset\belly_pain\bp-2.3gp → C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset_WAV\belly_pain\bp-2.wav
✅ Converted: C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset\belly_pain\bp-21.3gp → C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset_WAV\belly_pain\bp-21.wav
✅ Converted: C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset\belly_pain\bp-22.3gp → C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset_WAV\belly_pain\bp-22.wav
✅ Converted: C:\Users\Hussein\Desktop\baby_cries_language\Baby_Dataset\belly_pain\bp-23.3gp → C:\Users\Hussein\Deskto